In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [13]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from aiida_user_addons.process.transform import make_vac, make_supercell, rattle, get_primitive
from aiida_user_addons.tools.pymatgen import load_mp_struct

In [8]:
basepath = GroupPathX('mct-defect')
workpath = basepath['workflows']
elemental_path = basepath['elemental']

In [11]:
elemental_path['Hg_bulk'] = workpath['hg_elemental'].node

/home/bonan/aiida_env/aiida-2.0/aiida-grouppathx/aiida_grouppathx/pathx.py:485: UserWarning: Overwriting alias: Te_bulk with: Hg_bulk for this group
  warnings.warn(f'Overwriting alias: {existing_name} with: {alias} for this group')


In [17]:
elemental_path['Te_bulk'] = load_node(663583)

In [19]:
from matchest.aiida_utils.workflows.simple_vac import SimpleVacancyWorkChain

In [20]:
builder = SimpleVacancyWorkChain.get_builder()

In [32]:
hgte = read('HgTe.cif')
upd = VaspRelaxUpdater(builder = builder.relax, ).apply_preset(orm.StructureData(ase=hgte), 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'ncore':8 , 'kpar': 4}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600*12, queue_name='xhhctdnormal')
upd.set_label('HgTe RELAX')
upd.set_relax_settings(algo='rd')
builder.elemental_group_path = orm.Str('mct-defect/elemental')
builder.supercell_dim = orm.List([2,2,2])
# Updated parameters for supercell calculation
builder.supercell_workchain_updates = orm.Dict(
    {'options': {
        'resources': {'tot_num_mpiprocs': 128, 'num_machines':2},
        'max_wallclock_seconds': 3600 * 12, 
        'queue_name': 'xhhctdnormal',
        },
     'incar': {'kpar': 2, 'ncore': 8}
    }
)

In [30]:
from aiida.engine import submit

In [41]:
submit(builder)

<WorkChainNode: uuid: 7aa447ac-d925-4e62-86af-744d92da78b2 (pk: 663833) (matchest.aiida_utils.workflows.simple_vac.SimpleVacancyWorkChain)>

In [37]:
node= load_node('1cbd3d10-3abf')

In [39]:
node.outputs.vacancy_formation_energies.V_Hg.get_dict()

{'E_Hg': -0.54967641666667,
 'V_Hg': 1.2686411233333,
 'E_vac': 1.2686411233333,
 'E_supercell': -140.00825871}

In [40]:
node.outputs.vacancy_formation_energies.V_Te.get_dict()

{'E_Te': -3.5230132366667,
 'V_Te': 2.7249258833334,
 'E_vac': 2.7249258833334,
 'E_supercell': -140.00825871}